# Reliable Collection
> 대기, 예외처리, 중복 방지와 저장을 통해 수집 작업을 안정화합니다.

2026년 데이터 파이프라인 학습 내용을 정리한 실습 노트입니다.

## 데이터 수집 자동화

### 안정적 수집과 저장

#### 수집 오류 유형

- 요소 없음 : CSS선택자가 틀렸거나 화면에 없음 > 예외처리
- 로딩 지연 : 페이지가 늦게 뜸 > 명시적 대기
- 클릭 실패 : 버튼이 아직 클릭 불가 > 명시적 대기
- 데이터 누락 : 일부 항목이 페이지에 없음 > 기본값 처리
- 중복 수집 : 같은 데이터가 포함 > 판다스 중복처리
- 저장 실패 : 파일 존재여부, 권한여부 > 파일명 변경, 예외처리

In [ ]:
try:    # 실행중 오류가 날 것 같은 소스코드
    price = input('가격입력 > ')
    print(int(price))
except Exception as e: # 실행중 오류가 발생하면 처리할 부분
    print(f'가격은 숫자만 입력할 수 있습니다.')
finally: # 실행중 오류가 나든 안나든 무조건 실행하는 부분
    print('프로그램 종료')

In [ ]:
import pandas as pd
import os

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
driver = webdriver.Chrome()

try:
    driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')  # 웹사이트 오픈
    print(driver.title)  # 웹사이트의 head 영역에 있는 탭이름(title) 가져오기
except Exception as e:
    print(f'실행 중 오류발생! : {e.args}')
finally:
    driver.quit()

#### find_element 문제점
- find_elements() : 실행중 오류 발생 X
- find_element() : 실행중 오류 발생 가능성 높음
    - 함수를 만드는 게 유리

In [ ]:
# 안전하게 요소 추출함수
def get_text_safe(parent, selector, default=''):
    try:
        return parent.find_element(By.CSS_SELECTOR, selector).text
    except:
        return default  # 내가 지정한 기본값이 그냥 리턴

In [ ]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get('https://www.danawa.com/')

wait.until(
    EC.visibility_of_element_located((By.ID, 'danawa_footer'))
)

In [ ]:
get_text_safe(driver, 'div.item > a .txt', 'None')

#### 명시적 대기와 타임 아웃

- 명시적 대기를 하더라도 시간을 초과하면 실행 중 오류가 발생
- 실행 중 오류 잡으려면 try ~ except 사용 필요

In [ ]:
from selenium.common.exceptions import TimeoutException

try:
    driver.get('https://www.naver.com/')
    wait.until(
        # 네이버는 footer가 아이디. #footer 
        EC.visibility_of_element_located((By.CSS_SELECTOR, '#footer'))
    )
    print('로딩 완료!')
except TimeoutException as e:
    print('로딩 실패! 타임오버')

#### 버튼 클릭 가능상태까지 대기

- 버튼, 링크가 화면에 나타나도 js가 로딩완료될때까지 클릭못하는 경우가 발생할 수 있음
- 클릭상태까지 대기

In [ ]:
enable_button = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, '#account a'))
)

enable_button.click()

#### 로그파일 저장

- 화면에 출력보다 로그파일 저장이 효율적

In [ ]:
import logging

logging.basicConfig(
    filename='scraping.log',
    level=logging.INFO,   # DEBUG, INFO, WARN, WARNING, ERROR, FATAL...
    encoding='utf-8'
)

In [ ]:
logging.info('수집시작')
logging.info('상품 데이터 수집완료')
logging.error('수집 중 오류 발생!!')

#### DB에 저장

- 이전 저장한 다나와 데이터 사용

In [ ]:
df = pd.read_csv('../data/danawa_게이밍PC.csv', encoding='utf-8')
df = df.drop(columns=["Unnamed: 0"], errors='ignore')

In [ ]:
df.info()

In [ ]:
df

In [ ]:
from datetime import datetime
from getpass import getpass

# SQLAlchemy
from sqlalchemy import create_engine, URL, Text, BigInteger, Integer

In [ ]:
df.head()

In [ ]:
# 저장할 테이블명 - danawa_gaming_pc_yymmdd
table_name = 'danawa_gaming_pc_' + datetime.now().strftime('%y%m%d')
print(table_name)

In [ ]:
# PostgreSQL 연결정보
db_url = URL.create(
    'postgresql+psycopg',
    username='postgres',
    password='123456',
    host='127.0.0.1',
    port=5432,
    database='postgres'
)

In [ ]:
engine = create_engine(db_url)

In [ ]:
df.info()

In [ ]:
# 테이블 생성하고 DataFrame 데이터 저장
df.to_sql(
    name=table_name,
    con=engine,
    if_exists='fail',
    index=False,
    dtype={
        'index': BigInteger(),
        'computer_name': Text(),
        'computer_spec': Text(),
        'price_spec': Text(),
        'price': Integer()
    }
)

engine.dispose() # 사용 후 해제

print(f'{len(df)}건을 저장했습니다.')

#### 스케줄링

- 정해진 시간마다 반복 수행하도록 하는 것
- OS 마다 방식이 상이
    - Windows : 작업 스케줄러
    - Linux : cron 
    - Python : schedule 라이브러리 가능


##### 윈도우 작업 스케줄러

- 윈도우키 > 작업 스케줄러 검색 후 실행
- `기본 작업 만들기...` 클릭
- 기본 작업 만들기 > 이름과 설명(옵션) 추가
- 트리거 > 작업을 매일로 선택
    - 작업 내용은 시작시간과 반복 횟수 등 상황에 맞게 변경
- 작업 > 프로그램 시작
    - 프로그램 스크립트 : .bat 파일 위치 선택
    - 시작위치 : D:\SourceBank\ai-datapipeline-2026\chapt04 와 같이 경로 입력
- 속성 창 오픈 
    - 보안 옵션 : Administrators(관리자 그룹)을 선택하고 사용자 변경
    - 가장 높은 수준의 권한으로 실행

